# Whisper LoRA 中英双语微调 — Colab 免费 GPU 版

复用 `HenryVarro666/kokoro-aws-demo` 里的 `train.py` / `evaluate_cer.py`,在 Colab 免费 **T4 GPU** 上跑出真实的 **CER/WER before→after** 简历数字。AWS GPU 配额拿不到时的等效路径。

**开始前**:菜单 `Runtime → Change runtime type → 选 T4 GPU`,然后从上往下逐个 cell 运行(Shift+Enter)。全程约 **40–60 分钟**。

产物:`./output/adapter`(LoRA adapter,几十 MB)+ 一张 2×2 表(中文 CER / 英文 WER,微调前后)。

## 0. 确认拿到 GPU(应显示 Tesla T4, 15360MiB)

In [ ]:
!nvidia-smi

## 1. 拉代码 + 装依赖
torch 在 Colab 已预装(带 CUDA),只装其余依赖。`requirements.txt` 里已锁 `datasets<3.0`(FLEURS 走脚本加载,3.x 会挂)。

> 若装完弹"需要 RESTART"或后续 import 报版本错:菜单 `Runtime → Restart session`,再从**本 cell** 往下重跑。

In [ ]:
!git clone -q https://github.com/HenryVarro666/kokoro-aws-demo
%cd kokoro-aws-demo/training
!pip install -q -r requirements.txt

## 2. 配置旋钮
默认值是为 T4 + 首次出数字调的(约 40–60 分钟跑完)。想要更强的简历数字就调大 `TRAIN_SAMPLES`/`EPOCHS`。

> **T4 显存吃紧**:large-v3 在 16GB 上是临界。若后面 train 那步报 `CUDA out of memory`:
> - 把 `MODEL` 换成 `openai/whisper-medium`(稳),或
> - 把 `BATCH` 降到 `2`。

In [ ]:
import os
os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '60'   # FLEURS 下载放宽超时

MODEL = 'openai/whisper-large-v3'   # T4 OOM 就换 'openai/whisper-medium'
TRAIN_SAMPLES = 800   # 每语言条数(中+英 = 2×)
EPOCHS = 2
BATCH = 4             # T4 OOM 就降到 2
EVAL_N = 100          # 每语言评测条数
print(MODEL, TRAIN_SAMPLES, EPOCHS, BATCH)

## 3. 基线评测(微调前)
中文出 **CER**、英文出 **WER**(脚本按语言自动选)。先下载 large-v3(~3GB)+ FLEURS(~4GB),首次较慢。

In [ ]:
!python evaluate_cer.py --model_id $MODEL --dataset_config cmn_hans_cn --language zh --samples $EVAL_N
!python evaluate_cer.py --model_id $MODEL --dataset_config en_us       --language en --samples $EVAL_N

## 4. 微调(中英双语 LoRA + 8-bit)
`--use_8bit` 在 GPU 上才生效(bitsandbytes)。会先打印可训练参数(~1%),然后 loss 应逐步下降。

In [ ]:
!python train.py --model_id $MODEL --use_8bit --train_samples $TRAIN_SAMPLES --epochs $EPOCHS --batch_size $BATCH --output_dir ./output

## 5. 微调后评测 —— 这就是简历数字
和第 3 步同样两条,加 `--adapter ./output/adapter`。和基线对比 = 你的 2×2 表。

In [ ]:
!python evaluate_cer.py --model_id $MODEL --adapter ./output/adapter --dataset_config cmn_hans_cn --language zh --samples $EVAL_N
!python evaluate_cer.py --model_id $MODEL --adapter ./output/adapter --dataset_config en_us       --language en --samples $EVAL_N

## 6. 保存 adapter(防 Colab 断线丢失)
两种都做:存到 Google Drive(持久)+ 打包下载到本地。

In [ ]:
# 6a. 存 Google Drive
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/whisper-lora && cp -r ./output/adapter /content/drive/MyDrive/whisper-lora/
print('saved to Drive: MyDrive/whisper-lora/adapter')

In [ ]:
# 6b. 打包下载到本地
!cd output && zip -qr /content/whisper-lora-adapter.zip adapter
from google.colab import files
files.download('/content/whisper-lora-adapter.zip')

## 7.(可选)合并 + CTranslate2 int8 → 接回 AWS `/transcribe`
把 LoRA 合并进基座、转成 int8,产出 faster-whisper 能在 **CPU** 上加载的模型。上传到 S3 后,把 AWS 服务的 `ASR_MODEL` 指向它即可热替换基座(零代码改动)。

> large-v3 合并后转换较慢/占内存,Colab 上能跑;只是想要 AWS 部署用的成品时才运行本节。

In [ ]:
!pip install -q ctranslate2
import torch
from peft import PeftModel
from transformers import WhisperForConditionalGeneration, WhisperProcessor
base = WhisperForConditionalGeneration.from_pretrained(MODEL, torch_dtype=torch.float16)
merged = PeftModel.from_pretrained(base, './output/adapter').merge_and_unload()
merged.save_pretrained('./merged')
WhisperProcessor.from_pretrained('./output/adapter').save_pretrained('./merged')
!ct2-transformers-converter --model ./merged --output_dir ./whisper-ct2 \
  --quantization int8 --copy_files tokenizer_config.json preprocessor_config.json
!cd . && zip -qr /content/whisper-ct2.zip whisper-ct2 && echo 'zipped whisper-ct2.zip'

## 完成 —— 怎么用这些结果

1. **简历数字**:把第 3 步(基线)和第 5 步(微调后)的四个数填成一张表 —— *中文 CER X%→Y%,英文 WER A%→B%*。
2. **接回 AWS**:第 7 节产出的 `whisper-ct2/` 传到 S3 → 设 `ASR_MODEL` 指向它 → `/transcribe` 就用上你微调的模型了(对照 `02_后训练篇` §4)。
3. **想要更强的数字**:回第 2 步调大 `TRAIN_SAMPLES`(如 2000)和 `EPOCHS`(如 3),重跑 4–5。

面试讲法:*"AWS GPU 配额一时没批,我用 Colab 免费 T4 跑通了同一套 LoRA 双语微调脚本,拿到了 CER/WER 提升,再把 CT2 量化后的模型接回 AWS 的 CPU 推理服务"* —— 训练/部署解耦,本身就是工程判断力的体现。